# EDA - Booking

In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent

INPUT_PATH = BASE_DIR / "data/raw/booking/booking_data.csv"

df = pd.read_csv(INPUT_PATH)

df.head()

,name,rating,description,address,url,city,latitude,longitude
0,Chambres d'hôtes La Rive - Le Mont Saint Michel,9.5,Chambres d'hôtes La Rive - Le Mont Saint Miche...,"7, Route de la Rive, 50170 Ardevon, France",https://www.booking.com/hotel/fr/chambres-d-ho...,Mont Saint Michel,48.610818,-1.484205
1,La Suite Nolyvan,9.3,La Suite Nolyvan in Saint Malo offers a one-be...,"7 Rue du Chanoine Edouard Lainé, Sillon, 35400...",https://www.booking.com/hotel/fr/la-suite-noly...,St Malo,48.656857,-1.996288
2,Les Sablons - Magnifique Vue Port,9.5,Les Sablons - Magnifique Vue Port in Saint Mal...,"22 rue Dauphine, Saint-Servan, 35400 Saint Mal...",https://www.booking.com/hotel/fr/les-sablons-m...,St Malo,48.639125,-2.018749
3,Grand Hotel de Courtoisville,9.4,Discover the rebirth of a legendary address: a...,"9, rue Michelet, Sillon, 35400 Saint Malo, France",https://www.booking.com/hotel/fr/grand-de-cour...,St Malo,48.658402,-1.997361
4,Résidence Hostin'Malo - Au cœur de Saint Malo,9.0,Résidence Hostin'Malo in Saint Malo offers a 2...,"9 Avenue Jean Jaurès, 35400 Saint Malo, France",https://www.booking.com/hotel/fr/residence-mal...,St Malo,48.648603,-2.007690


## Duplicates

In [2]:
df_dup = df[df.duplicated(subset=["url"])]
print(df_dup)

Empty DataFrame
Columns: [name, rating, description, address, url, city, latitude, longitude]
Index: []


The output shows that the dataset currently contains no duplicated hotel URLs.

However, duplicates may appear in future scraping iterations because:
- the same hotel can appear multiple times across searches
- scraping retries may duplicate rows
- Booking pages may overlap between city queries

To make the ETL pipeline more robust, we remove potential duplicates using the hotel URL as business identifier.

In [3]:
df = df.drop_duplicates(subset=["url"])

## Numeric conversions

In [4]:
numeric_cols = ["rating", "latitude", "longitude"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[["rating", "latitude", "longitude"]].dtypes

rating       float64
latitude     float64
longitude    float64
dtype: object

The rating and coordinate columns are currently stored as object types.

Converting them to numeric types allows:
- numerical aggregations
- sorting and filtering
- PostgreSQL compatibility
- future geospatial analysis

## Missing values

In [5]:
df.isna().sum()

name           0
rating         2
description    0
address        0
url            0
city           0
latitude       0
longitude      0
dtype: int64

Some missing values are acceptable depending on the business meaning of the column.

Critical columns:
- hotel_name
- city
- url

Rows missing these fields are removed because they are necessary for joins and warehouse integrity.

Optional descriptive columns are filled with placeholder values.

In [8]:
df = df.rename(columns={"name": "hotel_name"})

critical_cols = ["hotel_name", "city", "url"]

df = df.dropna(subset=critical_cols)

median_rating = df["rating"].median()
df["rating"] = df["rating"].fillna(median_rating)

df["description"] = df["description"].fillna("No description available")
df["address"] = df["address"].fillna("Address unavailable")

## Text cleaning

In [9]:
text_cols = [
    "hotel_name",
    "city",
    "description",
    "address",
    "url"
]

for col in text_cols:
    df[col] = df[col].str.strip()

df["city"] = df["city"].str.title()

Scraped datasets often contain inconsistent whitespaces or city naming formats.

Standardizing text fields helps avoid issues during:
- merges
- groupby operations
- SQL joins
- dashboard filtering

## Final validation

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 354 entries, 0 to 353
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   hotel_name   354 non-null    str    
 1   rating       354 non-null    float64
 2   description  354 non-null    str    
 3   address      354 non-null    str    
 4   url          354 non-null    str    
 5   city         354 non-null    str    
 6   latitude     354 non-null    float64
 7   longitude    354 non-null    float64
dtypes: float64(3), str(5)
memory usage: 22.3 KB


In [11]:
df.duplicated(subset=["url"]).sum()

np.int64(0)

The cleaned dataset is now standardized and ready for:
- merging with weather data using the city column
- upload to S3 curated layer
- loading into PostgreSQL